# Coding Challenge Data Engineering

## Benötigte Imports

In [173]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, rank, regexp_extract, countDistinct, collect_list, concat
from pyspark.sql.window import Window

## SparkSession erstellen

In [174]:
spark = SparkSession.builder \
    .appName("DE_Challenge") \
    .getOrCreate()

## Einlesen der Daten

Die Daten aus dem Ordner data einlesen. Zur überprüfung ob dies geklappt hat wurden kurze prints/show erstellt.

In [175]:
# Einlesen der Dateien
products = spark.read.json("data/products.json")
stores_v1 = spark.read.csv("data/stores.csv", header=True, inferSchema=True)
stores_v2 = spark.read.csv("data/stores_v2.csv", header=True, inferSchema=True)
ticketline = spark.read.csv("data/ticket_line.csv", header=True, inferSchema=True)

# Überprüfen ob das Einlesen erfolgreich war
#products.show()
#stores_v1.show()
#stores_v2.show()
#ticketline.show()


## Schema der Stores anpassen

Da die store_id der Stores unterschiedliche Datentypen sind, muss dieser angepasst werden sodass kein Error beim joinen der Stores vorkommt

In [176]:
stores_v1_cleaned = stores_v1.select(
    concat(col("country"),(col("store_id")).cast("string")).alias("store_id"),
    col("country"),
    col("version")
)

stores_v2_cleaned = stores_v2.select(
    col("store_id"),
    # regexp_extract um die Kürzel des Landes rauszuziehen und als col country zu speichern für vereinheitlichung
    regexp_extract(col("store_id"), r"([A-Z]+)", 1).alias("country"),
    col("version")
)

stores_v1_cleaned.printSchema()
stores_v2_cleaned.printSchema()
stores_v1_cleaned.show()
stores_v2_cleaned.show()

# union by name da das schema angepasst wurde und duplikate können anschließend gedropped werden
stores_combined = (
    stores_v2_cleaned
    .unionByName(stores_v1_cleaned)
    .dropDuplicates(["store_id"])
)

stores_combined.show()



root
 |-- store_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- version: integer (nullable = true)

root
 |-- store_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- version: integer (nullable = true)

+--------+-------+-------+
|store_id|country|version|
+--------+-------+-------+
|    DE40|     DE|      1|
|    DE41|     DE|      1|
|    DE42|     DE|      1|
|    DE43|     DE|      1|
|    DE44|     DE|      1|
|    DE45|     DE|      1|
+--------+-------+-------+

+--------+-------+-------+
|store_id|country|version|
+--------+-------+-------+
|    DE45|     DE|      2|
|    DE46|     DE|      2|
|    DE47|     DE|      2|
+--------+-------+-------+

+--------+-------+-------+
|store_id|country|version|
+--------+-------+-------+
|    DE40|     DE|      1|
|    DE41|     DE|      1|
|    DE42|     DE|      1|
|    DE43|     DE|      1|
|    DE44|     DE|      1|
|    DE45|     DE|      2|
|    DE46|     DE|      2|
|    DE47|     DE| 

## Check ob die stores teil des Lidl Plus Programms sind

Da nicht alle stores Teil des Lidl Plus Programms sind muss dies überprüft werden. Nur die Stores welche in stores_v1 und stores_v2 enthalten sind sollen berücksichtigt werden und sind Teil des Programms

In [177]:
storesInLidlPlus = stores_combined.withColumn(
    "store_id_int",
    regexp_extract(col("store_id"), r"(\d+)$", 1).cast("integer")
).select(col("store_id_int")).distinct().orderBy("store_id", ascending = True)

storesInLidlPlus.show()

+------------+
|store_id_int|
+------------+
|          40|
|          41|
|          42|
|          43|
|          44|
|          45|
|          46|
|          47|
+------------+



## Q1 - Berechnung in wie vielen versch. Geschäften jedes Produkt verkauft wird

- Hierzu wurde vorab eine Filterung mittels eines inner-joins vorgenommen, um ticketline so zu filtern, dass nur die relevanten stores berücksichtigt werden.
- productStoreQuantitiy ist das df welches die Q1 beantwortet. Es wurde ein left-join mit products und ticketline_filtered durchgeführt um übersichtlich darzustellen in wie vielen verschiedenen Geschäften jedes Produkt verkauft wurde.
- Gruppierung nach product_id. Aggregiert und countDistinct genutzt um jeden Store nur ein mal zu zählen

In [186]:
ticketline_filtered = (
    ticketline.join(storesInLidlPlus, ticketline.store_id == storesInLidlPlus.store_id_int, how="inner")
    .drop(storesInLidlPlus.store_id_int)
)

productStoreQuantitiy = (
products.join(ticketline_filtered, on= "product_id", how = "left")
.groupBy("product_id", "product_name", "store_id")
.agg(countDistinct("store_id").alias("number_of_stores"))
)
productStoreQuantitiy.show()


+----------+--------------------+--------+----------------+
|product_id|        product_name|store_id|number_of_stores|
+----------+--------------------+--------+----------------+
|         2|      Milbona cheese|      43|               1|
|         6|      Sondey cookeys|      42|               1|
|         4|    Crownfield black|      46|               1|
|         4|    Crownfield black|      42|               1|
|         4|    Crownfield black|      45|               1|
|         4|    Crownfield black|      41|               1|
|         5|Crownfield with milk|      45|               1|
|         3|       Milbona yogur|      45|               1|
|         6|      Sondey cookeys|      43|               1|
|         6|      Sondey cookeys|      44|               1|
|         9|      Freshona beans|      43|               1|
|         1|        Milbona milk|      46|               1|
|         4|    Crownfield black|      44|               1|
|         9|      Freshona beans|      4

## Q2 - Store mit dem zweithöchsten Gesamtverkauf pro Produkt

- Zuerst wird die Gesamtmenge total_quantity pro Produkt und Store berechnet, indem alle Einzelverkäufe aus ticketline_filtered summiert werden.
- Mittels einer Window Function wird pro Produkt ein Rang vergeben. partitionBy("product_id") sorgt dafür dass der Rang pro Produkt neu gestartet wird. orderBy(total_quantity.desc()) gibt dem Store mit der höchsten Menge Rang 1. rank() wurde bewusst statt row_number() gewählt
- Nur Rang 2 wird behalten

### Besonderheit

- Bei Gleichstand auf Rang 2 (zwei Stores mit identischer Menge) gibt rank() beiden denselben Rang. row_number() würde zufällig einen raussuchen.
- Produkte in nur einem Store haben keinen Rang 2 und werden aus dem Ergebnis ausgeschlossen.

In [191]:
productStoreQuantitiyRank = (
    ticketline_filtered
    .groupBy("product_id", "store_id")
    .agg(sum("quantity").alias("total_quantity"))
)

window_spec = Window.partitionBy("product_id").orderBy(col("total_quantity").desc())
ranked = productStoreQuantitiyRank.withColumn("rank", rank().over(window_spec))

second_store = ranked.filter(col("rank") == 2)

second_store.show()

+----------+--------+--------------+----+
|product_id|store_id|total_quantity|rank|
+----------+--------+--------------+----+
|         1|      42|            31|   2|
|         2|      47|            26|   2|
|         3|      44|            28|   2|
|         4|      46|            36|   2|
|         5|      41|            24|   2|
|         7|      40|            36|   2|
|         8|      47|            29|   2|
|         9|      40|            23|   2|
|         9|      47|            23|   2|
|        10|      46|            31|   2|
+----------+--------+--------------+----+



## Q3 - Rank 2 Stores gruppiert nach Kategorie

- Das categories-Feld in products ist ein Array von Structs. Mit col("categories.category_name") werden direkt alle Kategorienamen aus dem Array extrahiert.
- second_store aus der vorherigen Aufgabe (Q2) wird mit den Produktkategorien per inner-join auf product_id verknüpft. So weiß man für jeden Second Store zu welcher Kategorie das Produkt gehört.
- groupBy("category_name") fasst alle Zeilen derselben Kategorie zusammen. collect_list("store_id") sammelt alle store_ids einer Kategorie in eine Liste

In [192]:
products_with_categories = products.select(
    col("product_id"),
    col("categories.category_name")
)

result_by_category = (
    second_store
    .join(products_with_categories, on="product_id", how="inner")
    .groupBy("category_name")
    .agg(collect_list("store_id").alias("second_stores"))
    .orderBy("category_name")
)

result_by_category.show()

+-----------------+-------------+
|    category_name|second_stores|
+-----------------+-------------+
|         [Cereal]|     [46, 41]|
|        [Cookies]|         [47]|
|[Cookies, Cereal]|         [40]|
|  [Dairy product]| [42, 47, 44]|
|     [Vegetables]| [40, 47, 46]|
+-----------------+-------------+



## Q4 - ETL Datapipeline in Produktion bringen

### Versionskontrolle
Der Code sollte am besten in einem Git-Repository vewaltet werden. Wenn Änderungen stattfinden sollen, macht es Sinn sich einen feature Branch zu erstellen in dem man die Änderungen vornimmt und schaut ob alles passt. Anschließend wird eine Pull-Request erstellt und von min. einer weiteren Person geschaut ob die Änderung den Anforderungen entspricht. So wird sichergestellt dass keineunkontrollierten Änderungen in Production gelangen.

Besonders zu achten ist auf sensible Daten, Passwörter oder API-Keys. Diese nie im Code speichern sondern in der .gitignore File.

### Deployment-Automatisierung
Ein CI/CD-System übernimmt das automatische Deployment. Durch meine Werkstudententätigkeit weiß ich dass über die Scheduler die Zeiten des Deployments geregelt/getriggert werden. Diese sind individuell anpassbar.

### Regressionsschutz
Um sicherzustellen dass neue Releases keine bestehende Funktionalität beeinträchtigen, müssen automatisierte Tests statfinden.

- Einzelne Transformationsschritte werden mit kleinen Dummy-dfs getestet, ob beispielsweise die Window Function bei Ties das erwartete Ergebnis liefert.
- Der gesamte ETL-Prozess wird mit einem kleinen Testdatensatz durchgeführt und das Output gegen ein erwartetes Ergebnis geprüft.
- Datenchecks nach jedem Step. Bei Fehler schlägt der Job fehl anstatt Datenfehler unbemerkt weiterzuleiten.

Die Tests laufen automatisch im CI/CD-System bei jedem Pull Request. So ist ein Merge nur möglich wenn alle Tests grün sind und es keine auffälligkeiten gibt.

### Monitoring

Monitoring ist essenziell um zu schauen ob die Pipelines richtig laufen.

- Logging durch Log-Einträge für die Nachvolziehbarkeit. Wann und wo ist der Fehler aufgetreten?
- Bei Job-Fehler oder auffälligen Metriken wird automatisch eine Benachrichtigung verschickt und man bekommt einen Alert in bspw. Databricks dass eine pipeline gefailed ist.